# Tree models
Week 5: a decision tree and a random forest, compared with the Week 4 Linear Regression baseline (`lr_zip`, validation MdAPE 9.52%).
CRISP-DM Modeling phase: same test design as 03, then build, describe and assess.

Lessons applied from the week 5 meeting (`week5-notes.md`):
- same splits and the same `idx_pipeline.py` preprocessing for every model, or the comparison means nothing
- regularize trees with depth and leaf size, and always report the train vs validation gap
- choose on MdAPE / MAPE, not R²
- break errors down by price band and county

#### Setup

In [1]:
import json
import os
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

# shared with 03 and 05: loading, Dataset A fills (fit on train only), encoding and metrics
from idx_pipeline import CAT_COLS, DATA_DIR, SEED, TARGET, build_preprocessor, error_by, load_split, metrics

np.random.seed(SEED)
MODEL_DIR = os.path.join(DATA_DIR, "models")  # inside dataset/, which .gitignore excludes
os.makedirs(MODEL_DIR, exist_ok=True)

#### Test design (written before training)
- **Split** (handbook §01): train = Jan 2024 – Feb 2026, validation = Mar 2026, test = Apr 2026. Same as 03.
- **Choices** (depth, leaf size, log target) are made on **validation** only.
- **Test is not used here.** It's scored once in Week 8 with every model together. The team's week 5 table reports test scores; choosing models on test makes the final numbers look better than they will be on new data.
- **Metrics** (handbook §08): MdAPE is the headline; MAPE, R², MAE and the share within 10% alongside. Train scores too, for the overfitting gap.
- **Success criterion**: beat `lr_zip` on validation MdAPE by more than noise, without a large train/validation gap.

#### Step 1: Load the cleaned splits

In [2]:
data = {s: load_split(s) for s in ["train", "val"]}  # test is held out until Week 8

# same feature list as 03, from the dataset description made in 02
description = pd.read_csv(os.path.join(DATA_DIR, "dataset_description.csv"))
features = description.loc[description["role"].isin(["model_feature", "missing_flag"]), "column"].tolist()
# trees split on raw values, so nothing is logged or scaled; categoricals become integer codes
NUM_COLS = [c for c in features if c not in CAT_COLS]

X = {s: d[features] for s, d in data.items()}
y = {s: d[TARGET].to_numpy() for s, d in data.items()}

baseline = pd.read_csv(os.path.join(MODEL_DIR, "baseline_metrics.csv"), index_col=0)  # from 03
{s: d.shape for s, d in data.items()}

{'train': (282279, 42), 'val': (11412, 42)}

In [3]:
def build_tree_model(estimator, log_target=False):
    # Dataset A fills + integer codes for City / county / ZIP, then the tree
    pipe = Pipeline([("pre", build_preprocessor("tree", NUM_COLS, CAT_COLS)), ("model", estimator)])
    if log_target:
        # fit log(price), predict exp(): the model cares about % error instead of $ error
        return TransformedTargetRegressor(regressor=pipe, func=np.log, inverse_func=np.exp)
    return pipe


def evaluate(model):
    # fit on train only; score validation and train, and the gap between them
    start = time.time()
    model.fit(X["train"], y["train"])
    val_pred = model.predict(X["val"])
    tr, va = metrics(y["train"], model.predict(X["train"])), metrics(y["val"], val_pred)
    row = {**{f"val_{k}": v for k, v in va.items()},
           "train_MdAPE_%": tr["MdAPE_%"], "train_R2": tr["R2"],
           "gap_MdAPE": round(va["MdAPE_%"] - tr["MdAPE_%"], 2),  # how much worse on validation
           "gap_R2": round(tr["R2"] - va["R2"], 4),
           "fit_s": round(time.time() - start)}
    return row, val_pred

#### Step 2: Decision tree
A few `(max_depth, min_samples_leaf)` pairs, including the ones Jihad (15, 10) and Stephanie (12, 20) used. `(None, 1)` is sklearn's default: the tree grows until every leaf is one home, which should overfit.

In [4]:
DT_GRID = [(None, 1), (8, 20), (12, 20), (15, 10), (20, 5)]  # (max_depth, min_samples_leaf)

dt_rows, dt_params = {}, {}
for depth, leaf in DT_GRID:
    name = f"dt_depth{depth}_leaf{leaf}"
    dt_params[name] = dict(max_depth=depth, min_samples_leaf=leaf)
    dt_rows[name], _ = evaluate(build_tree_model(DecisionTreeRegressor(random_state=SEED, **dt_params[name])))

dt_results = pd.DataFrame(dt_rows).T
dt_results

,val_MdAPE_%,val_MAPE_%,val_R2,val_MAE_$,val_within_10%,train_MdAPE_%,train_R2,gap_MdAPE,gap_R2,fit_s
dt_depthNone_leaf1,10.67,16.35,0.7870,213925.0,47.8,0.00,1.0000,10.67,0.2130,16.0
dt_depth8_leaf20,18.21,26.50,0.7017,299467.0,29.4,17.93,0.7306,0.28,0.0289,7.0
dt_depth12_leaf20,12.00,17.32,0.8132,215239.0,43.6,11.45,0.8530,0.55,0.0398,7.0
dt_depth15_leaf10,10.10,15.19,0.8364,194161.0,49.7,9.01,0.8999,1.09,0.0635,4.0
dt_depth20_leaf5,9.39,14.39,0.8363,189504.0,52.3,6.29,0.9437,3.10,0.1074,5.0


In [5]:
# best tree on validation MdAPE, then the same tree with a log target
best_dt = dt_results["val_MdAPE_%"].idxmin()
dt_rows[f"{best_dt}_log"], _ = evaluate(
    build_tree_model(DecisionTreeRegressor(random_state=SEED, **dt_params[best_dt]), log_target=True))
dt_results = pd.DataFrame(dt_rows).T
print("best decision tree on validation:", best_dt)
dt_results.loc[[best_dt, f"{best_dt}_log"]]

best decision tree on validation: dt_depth20_leaf5


,val_MdAPE_%,val_MAPE_%,val_R2,val_MAE_$,val_within_10%,train_MdAPE_%,train_R2,gap_MdAPE,gap_R2,fit_s
dt_depth20_leaf5,9.39,14.39,0.8363,189504.0,52.3,6.29,0.9437,3.10,0.1074,5.0
dt_depth20_leaf5_log,9.35,14.12,0.8379,187914.0,52.4,6.03,0.9405,3.32,0.1026,5.0


#### Step 3: Random forest
200 trees, all cores. Two leaf sizes, each with and without a log target.
A full-depth forest (`min_samples_leaf=1`) is about 5 GB in memory, so only the best forest so far is kept.

In [6]:
RF_VARIANTS = {"rf_leaf5": dict(min_samples_leaf=5), "rf": dict(min_samples_leaf=1)}  # small one first

rf_rows, rf_config, val_preds = {}, {}, {}
best_rf_name, best_rf_model = None, None
for base, params in RF_VARIANTS.items():
    for log_target in [False, True]:
        name = base + ("_log" if log_target else "")
        rf_config[name] = (params, log_target)
        model = build_tree_model(RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=SEED, **params),
                                 log_target)
        rf_rows[name], val_preds[name] = evaluate(model)
        if best_rf_name is None or rf_rows[name]["val_MdAPE_%"] < rf_rows[best_rf_name]["val_MdAPE_%"]:
            best_rf_name, best_rf_model = name, model
        del model

rf_results = pd.DataFrame(rf_rows).T
print("best random forest on validation:", best_rf_name)
rf_results

best random forest on validation: rf


,val_MdAPE_%,val_MAPE_%,val_R2,val_MAE_$,val_within_10%,train_MdAPE_%,train_R2,gap_MdAPE,gap_R2,fit_s
rf_leaf5,7.76,11.99,0.8879,156145.0,59.7,4.81,0.9477,2.95,0.0598,189.0
rf_leaf5_log,7.76,11.38,0.8862,154047.0,60.2,4.74,0.9447,3.02,0.0585,122.0
rf,7.72,11.99,0.8919,154024.0,60.0,2.71,0.9852,5.01,0.0933,157.0
rf_log,7.75,11.32,0.8918,151211.0,60.0,2.69,0.9819,5.06,0.0901,158.0


#### Step 4: Compare with the baseline

In [7]:
# lr_zip from 03, in the same columns
lr, lr_train = baseline.loc["lr_zip"], baseline.loc["lr_zip (train)"]
lr_row = {**{f"val_{k}": lr[k] for k in lr.index}, "train_MdAPE_%": lr_train["MdAPE_%"], "train_R2": lr_train["R2"],
          "gap_MdAPE": round(lr["MdAPE_%"] - lr_train["MdAPE_%"], 2), "gap_R2": round(lr_train["R2"] - lr["R2"], 4)}

comparison = pd.concat([pd.DataFrame({"lr_zip (03)": lr_row}).T,
                        dt_results.loc[[best_dt, f"{best_dt}_log"]],
                        rf_results]).sort_values("val_MdAPE_%")
comparison

,val_MdAPE_%,val_MAPE_%,val_R2,val_MAE_$,val_within_10%,train_MdAPE_%,train_R2,gap_MdAPE,gap_R2,fit_s
rf,7.72,11.99,0.8919,154024.0,60.0,2.71,0.9852,5.01,0.0933,157.0
rf_log,7.75,11.32,0.8918,151211.0,60.0,2.69,0.9819,5.06,0.0901,158.0
rf_leaf5,7.76,11.99,0.8879,156145.0,59.7,4.81,0.9477,2.95,0.0598,189.0
rf_leaf5_log,7.76,11.38,0.8862,154047.0,60.2,4.74,0.9447,3.02,0.0585,122.0
dt_depth20_leaf5_log,9.35,14.12,0.8379,187914.0,52.4,6.03,0.9405,3.32,0.1026,5.0
dt_depth20_leaf5,9.39,14.39,0.8363,189504.0,52.3,6.29,0.9437,3.10,0.1074,5.0
lr_zip (03),9.52,13.56,0.8298,181042.0,51.8,9.29,0.8582,0.23,0.0284,NaN


#### Step 5: Error breakdown on validation (handbook §08)
Best forest vs `lr_zip`, by price band (quintiles of the actual price) and by the 10 largest counties.

In [8]:
val = data["val"]
lr_pred = joblib.load(os.path.join(MODEL_DIR, "baseline_lr.joblib")).predict(X["val"])
rf_pred = val_preds[best_rf_name]
cols = ["MdAPE_%", "MAPE_%", "within_10%"]

bands = pd.qcut(val[TARGET], 5, labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"])
by_band = val.assign(price_band=bands)
pd.concat({"lr_zip": error_by(by_band, lr_pred, "price_band")[["n"] + cols],
           best_rf_name: error_by(by_band, rf_pred, "price_band")[cols]}, axis=1)

/Users/emilychen/mambaforge/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.9.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/emilychen/mambaforge/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/emilychen/mambaforge/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.9.1 when using v

lr_zip                                rf                  
                 n MdAPE_% MAPE_% within_10% MdAPE_% MAPE_% within_10%
Q1 lowest   2283.0    9.42  15.37       52.5    7.11  13.78       61.1
Q2          2293.0    7.99  11.64       58.6    5.90   9.82       69.6
Q3          2272.0    8.38  11.57       56.4    6.58  10.28       66.5
Q4          2284.0    9.49  12.55       52.3    8.80  11.95       56.3
Q5 highest  2280.0   13.33  16.68       39.4   10.73  14.12       46.5

In [9]:
top_counties = val["CountyOrParish"].value_counts().index[:10]
in_top = val["CountyOrParish"].isin(top_counties).to_numpy()
pd.concat({"lr_zip": error_by(val[in_top], lr_pred[in_top], "CountyOrParish")[["n"] + cols],
           best_rf_name: error_by(val[in_top], rf_pred[in_top], "CountyOrParish")[cols]}, axis=1
          ).sort_values(("lr_zip", "n"), ascending=False)

lr_zip                                rf                  
                      n MdAPE_% MAPE_% within_10% MdAPE_% MAPE_% within_10%
Los Angeles      2785.0   10.00  13.90       50.0    8.67  12.83       56.0
Riverside        1917.0    9.37  14.54       52.6    6.33  11.44       65.3
San Diego        1273.0    9.35  12.77       52.1    7.71  10.66       62.1
San Bernardino   1196.0    7.84  11.77       59.8    6.25  10.61       67.6
Orange           1099.0   10.23  12.89       49.1    8.15  11.29       59.3
Contra Costa      538.0    8.54  12.46       55.0    7.77  11.26       60.8
Alameda           491.0   11.20  15.40       44.2    9.66  13.38       52.5
Ventura           369.0    8.55  13.93       54.2    7.17  10.80       62.6
Santa Clara       337.0    9.37  12.30       52.5    8.59  10.87       57.3
San Luis Obispo   202.0   11.89  14.45       45.5    8.63  13.26       55.0

#### Step 6: What the forest relies on
Permutation importance on **validation**: shuffle one input column and measure how much MdAPE gets worse. It's more trustworthy than the forest's built-in importance, which favors columns with many distinct values.
Location columns (lat/long, ZIP, City, county) carry overlapping information (Rohan's VIF ≈ 18), so shuffling one alone understates how much location matters in total.

In [10]:
mdape = make_scorer(lambda yt, yp: np.median(np.abs(yp - yt) / yt), greater_is_better=False)
pi = permutation_importance(best_rf_model, X["val"], y["val"], scoring=mdape, n_repeats=3, random_state=SEED)
importance = (pd.DataFrame({"MdAPE_increase_pts": 100 * pi.importances_mean, "std": 100 * pi.importances_std},
                           index=X["val"].columns)
                .sort_values("MdAPE_increase_pts", ascending=False).round(2))
importance.head(15)

,MdAPE_increase_pts,std
Longitude,15.29,0.22
Latitude,14.48,0.22
PostalCode,11.82,0.07
LivingArea,11.12,0.21
CountyOrParish,5.68,0.14
LotSizeSquareFeet,1.70,0.06
City,1.02,0.02
BathroomsTotalInteger,0.94,0.03
YearBuilt,0.69,0.02
AgeAtSale,0.65,0.03


#### Step 7: Save the best forest and results

In [11]:
joblib.dump(best_rf_model, os.path.join(MODEL_DIR, "trees_rf.joblib"), compress=3)
params, log_target = rf_config[best_rf_name]
with open(os.path.join(MODEL_DIR, "trees_rf_features.json"), "w") as f:
    json.dump({"model": best_rf_name, "features": features, "categorical": CAT_COLS, "n_estimators": 200,
               **params, "log_target": log_target, "seed": SEED}, f, indent=2)
comparison.to_csv(os.path.join(MODEL_DIR, "trees_metrics.csv"))  # validation + train, no test
importance.to_csv(os.path.join(MODEL_DIR, "trees_permutation_importance.csv"))
print("saved to", MODEL_DIR, f"({os.path.getsize(os.path.join(MODEL_DIR, 'trees_rf.joblib')) / 1e6:,.0f} MB model)")

saved to dataset/processed/models (969 MB model)


#### Step 8 (optional): 12-month training window
Same best forest, trained on Mar 2025 – Feb 2026 only.

In [12]:
del best_rf_model  # free memory before fitting another forest
recent = (data["train"]["source_month"] >= "2025-03").to_numpy()
X_full, y_full = X["train"], y["train"]
X["train"], y["train"] = X_full[recent], y_full[recent]
params, log_target = rf_config[best_rf_name]
window_row, _ = evaluate(build_tree_model(
    RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=SEED, **params), log_target))
X["train"], y["train"] = X_full, y_full  # back to the full 26 months

pd.DataFrame({f"{best_rf_name} (26 months)": rf_rows[best_rf_name],
              f"{best_rf_name} (12 months)": window_row}).T.assign(train_rows=[len(X_full), int(recent.sum())])

,val_MdAPE_%,val_MAPE_%,val_R2,val_MAE_$,val_within_10%,train_MdAPE_%,train_R2,gap_MdAPE,gap_R2,fit_s,train_rows
rf (26 months),7.72,11.99,0.8919,154024.0,60.0,2.71,0.9852,5.01,0.0933,157.0,282279
rf (12 months),8.05,12.85,0.8773,164567.0,58.0,2.84,0.9833,5.21,0.1060,60.0,128031
